# Three-Variable Panel VAR: Utilization x Volatility x Liquidations

**Panel:** 30 CSUs across 10+ chains, ~660 daily observations (Jan 1 2024 - Dec 31 2025)  
**Variables:** Utilization (leverage proxy), Collateral basket volatility (14-day rolling), Liquidations (log(1 + n_liquidations))  
**Method:** Fixed-effects VAR with Cholesky identification, bootstrap confidence intervals  
**Ordering:** Utilization -> Volatility -> Liquidation (slow to fast)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.api import VAR
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## 1. Load Panel Data

In [ ]:
df = pd.read_parquet('../data/analysis/panel_svar_data.parquet')
df['date'] = df['date'].astype(str)

print(f"Raw panel: {len(df):,} obs, {df['csu'].nunique()} CSUs")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Columns: {list(df.columns)}")
print(f"\nCoverage:")
for col in ['utilization', 'volatility', 'n_liquidations', 'liquidation']:
    cov = df[col].notna().mean() * 100
    print(f"  {col}: {cov:.1f}%")

## 2. Prepare Analysis Variables

We use three variables:
1. **Utilization** -- borrowed USD / supplied USD (leverage proxy)
2. **Volatility** -- 14-day rolling std of collateral basket log returns
3. **Liquidation** -- log(1 + n_liquidations) to handle zero-liquidation days

Drop rows where utilization or volatility is missing (markets not yet deployed).

In [ ]:
df_clean = df.dropna(subset=['utilization', 'volatility']).copy()

print(f"After dropping missing util/vol: {len(df_clean):,} obs, {df_clean['csu'].nunique()} CSUs")
print(f"Date range: {df_clean['date'].min()} to {df_clean['date'].max()}")
print(f"\nAll CSUs:")
for csu in sorted(df_clean['csu'].unique()):
    sub = df_clean[df_clean['csu'] == csu]
    liq_days = (sub['n_liquidations'] > 0).sum()
    print(f"  {csu}: {len(sub)} obs, util={sub['utilization'].mean():.3f}, "
          f"vol={sub['volatility'].mean():.4f}, liq_days={liq_days}")

## 3. Define Subsamples

In [ ]:
pooled_csus = [
    'aave_v3_arbitrum', 'aave_v3_avalanche', 'aave_v3_base',
    'aave_v3_binance', 'aave_v3_ethereum', 'aave_v3_linea',
    'aave_v3_optimism', 'aave_v3_polygon', 'aave_v3_scroll',
    'aave_v3_xdai',
    'benqi_lending_avalanche', 'moonwell_lending_base',
    'sparklend_ethereum', 'venus_core_pool_binance',
    'fluid_lending_arbitrum',
    'compound_v2_ethereum',
    'lodestar_lending_arbitrum', 'mendi_lending_linea',
]

isolated_csus = [
    'compound_v3_arb_usdc', 'compound_v3_arb_usdc_e',
    'compound_v3_arb_usdt', 'compound_v3_arb_weth',
    'compound_v3_base_aero', 'compound_v3_base_usdc',
    'compound_v3_base_usdbc', 'compound_v3_base_weth',
    'compound_v3_eth_usdc', 'compound_v3_eth_usds',
    'compound_v3_eth_usdt', 'compound_v3_eth_weth',
]

stablecoin_markets = [
    'compound_v3_arb_usdc', 'compound_v3_arb_usdc_e',
    'compound_v3_arb_usdt', 'compound_v3_base_usdc',
    'compound_v3_eth_usdc', 'compound_v3_eth_usds',
    'compound_v3_eth_usdt',
]

volatile_asset_markets = [
    'compound_v3_arb_weth', 'compound_v3_base_aero',
    'compound_v3_base_weth', 'compound_v3_eth_weth',
]

# Filter to CSUs present in clean data
valid = set(df_clean['csu'].unique())
pooled_csus = [c for c in pooled_csus if c in valid]
isolated_csus = [c for c in isolated_csus if c in valid]
stablecoin_markets = [c for c in stablecoin_markets if c in valid]
volatile_asset_markets = [c for c in volatile_asset_markets if c in valid]

samples = {
    'Full Sample': pooled_csus + isolated_csus,
    'Pooled': pooled_csus,
    'Isolated': isolated_csus,
    'Stablecoin': stablecoin_markets,
    'Volatile Asset': volatile_asset_markets,
}

for label, csus in samples.items():
    sub = df_clean[df_clean['csu'].isin(csus)]
    print(f"{label:15s}: {len(csus):2d} CSUs, {len(sub):,} obs, "
          f"liq events={sub['n_liquidations'].sum():,}")

## 4. Summary Statistics

In [ ]:
rows = []
for label, csus in samples.items():
    sub = df_clean[df_clean['csu'].isin(csus)]
    rows.append({
        'Sample': label,
        'N (CSUs)': sub['csu'].nunique(),
        'T (dates)': sub['date'].nunique(),
        'Obs': len(sub),
        'Util Mean': sub['utilization'].mean(),
        'Util SD': sub['utilization'].std(),
        'Vol Mean': sub['volatility'].mean(),
        'Vol SD': sub['volatility'].std(),
        'Liq/Day Mean': sub['n_liquidations'].mean(),
        'Liq/Day SD': sub['n_liquidations'].std(),
        '% Days w/ Liq': 100 * (sub['n_liquidations'] > 0).mean(),
    })

summary_df = pd.DataFrame(rows)
print("Table 1: Panel Summary Statistics")
print("=" * 110)
print(summary_df.to_string(index=False, float_format=lambda x: f"{x:.4f}" if abs(x) < 1 else f"{x:,.1f}"))

## 5. Panel VAR Estimation Function

**Procedure:**
1. Within-transform (demean by CSU) to remove fixed effects
2. Estimate VAR(p) on the demeaned panel
3. Select lag order via BIC (capped at 5)
4. Orthogonalized IRF via Cholesky: utilization -> volatility -> liquidation
5. 95% bootstrap CIs (500 replications, residual resampling)

In [ ]:
VAR_COLS = ['utilization', 'volatility', 'liquidation']
FE_COLS = ['util_fe', 'vol_fe', 'liq_fe']
VAR_LABELS = {'util_fe': 'Utilization', 'vol_fe': 'Volatility', 'liq_fe': 'Liquidation'}
IDX_MAP = {0: 'util', 1: 'vol', 2: 'liq'}  # Cholesky order

def run_panel_var_3v(df, csus, label, max_lags=10, irf_periods=20, n_boot=500):
    """
    Run 3-variable fixed-effects Panel VAR.
    Returns dict with results, IRFs, bootstrap CIs.
    """
    sub = df[df['csu'].isin(csus)].copy()
    sub = sub.dropna(subset=VAR_COLS)
    
    # Fixed effects: within-transformation (demean by CSU)
    for orig, fe in zip(VAR_COLS, FE_COLS):
        sub[fe] = sub.groupby('csu')[orig].transform(lambda x: x - x.mean())
    
    panel = sub[FE_COLS].dropna()
    print(f"  Panel: {len(panel):,} obs after FE transform")
    
    # Fit VAR with BIC-selected lags
    model = VAR(panel)
    try:
        lo = model.select_order(maxlags=min(max_lags, len(panel) // 10))
        optimal = max(1, min(lo.bic, 5))
        print(f"  Lag selection -- AIC: {lo.aic}, BIC: {lo.bic}, HQIC: {lo.hqic} -> using {optimal}")
    except:
        optimal = 2
        print(f"  Lag selection failed, using {optimal}")
    
    results = model.fit(optimal)
    
    # Orthogonalized IRFs
    irf = results.irf(irf_periods)
    irf_orth = irf.orth_irfs  # shape: (periods+1, 3, 3)
    
    # Bootstrap CIs (residual resampling)
    rng = np.random.RandomState(42)
    boot_irfs = []
    resids = results.resid.values
    fitted = results.fittedvalues.values
    
    for _ in range(n_boot):
        idx = rng.choice(len(resids), size=len(resids), replace=True)
        boot_data = fitted + resids[idx]
        try:
            br = VAR(pd.DataFrame(boot_data, columns=panel.columns)).fit(optimal)
            boot_irfs.append(br.irf(irf_periods).orth_irfs)
        except:
            continue
    
    ci_lo = ci_hi = None
    if len(boot_irfs) > 50:
        ba = np.array(boot_irfs)
        ci_lo = np.percentile(ba, 2.5, axis=0)
        ci_hi = np.percentile(ba, 97.5, axis=0)
        print(f"  Bootstrap: {len(boot_irfs)}/{n_boot} successful replications")
    
    return {
        'label': label,
        'results': results,
        'irf_orth': irf_orth,
        'ci_lo': ci_lo,
        'ci_hi': ci_hi,
        'n_csus': sub['csu'].nunique(),
        'n_obs': len(sub),
        'lags': optimal,
    }

## 6. Run All Analyses

In [ ]:
all_results = {}
for label, csus in samples.items():
    print(f"\n{'='*60}")
    print(f"  {label} ({len(csus)} CSUs)")
    print(f"{'='*60}")
    all_results[label] = run_panel_var_3v(df_clean, csus, label)

## 7. IRF Plots -- Full 3x3 Grid (Full Sample)

In [ ]:
def plot_irf(res, imp_idx, resp_idx, title, ax, color='blue'):
    """Plot a single IRF path with bootstrap CI."""
    vals = res['irf_orth'][:, imp_idx, resp_idx]
    periods = range(len(vals))
    ax.plot(periods, vals, color=color, lw=2.5)
    if res['ci_lo'] is not None:
        ax.fill_between(periods,
                        res['ci_lo'][:, imp_idx, resp_idx],
                        res['ci_hi'][:, imp_idx, resp_idx],
                        alpha=0.2, color=color)
    ax.axhline(0, color='black', ls='--', lw=0.8)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Days')
    return ax

var_names = ['Utilization', 'Volatility', 'Liquidation']
res = all_results['Full Sample']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
for i in range(3):  # impulse
    for j in range(3):  # response
        title = f'{var_names[i]} -> {var_names[j]}'
        plot_irf(res, i, j, title, axes[j, i])
        if j == 0:
            axes[j, i].set_title(f'Shock: {var_names[i]}\n-> {var_names[j]}',
                                  fontsize=11, fontweight='bold')

fig.suptitle(f'Full Sample 3-Variable IRFs (N={res["n_csus"]}, {res["n_obs"]:,} obs, {res["lags"]} lags)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../results/irf_3var_full_grid.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

## 8. Key IRFs: Cross-Variable Responses by Subsample

In [ ]:
# The 6 cross-variable IRFs, compared across subsamples
cross_irfs = [
    (0, 1, 'Util -> Vol'),
    (0, 2, 'Util -> Liq'),
    (1, 0, 'Vol -> Util'),
    (1, 2, 'Vol -> Liq'),
    (2, 0, 'Liq -> Util'),
    (2, 1, 'Liq -> Vol'),
]

colors = {
    'Full Sample': 'black',
    'Pooled': '#3498db',
    'Isolated': '#e74c3c',
    'Stablecoin': '#2ecc71',
    'Volatile Asset': '#e67e22',
}

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
for ax, (imp, resp, title) in zip(axes.flat, cross_irfs):
    for label, res in all_results.items():
        vals = res['irf_orth'][:, imp, resp]
        c = colors.get(label, 'gray')
        lw = 2.5 if label == 'Full Sample' else 1.5
        ls = '-' if label in ('Full Sample', 'Pooled', 'Isolated') else '--'
        ax.plot(range(len(vals)), vals, color=c, lw=lw, ls=ls, label=label)
    ax.axhline(0, color='black', ls='--', lw=0.5)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Days')
    ax.legend(fontsize=8)

fig.suptitle('Cross-Variable IRFs by Subsample (3-Variable PVAR)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/irf_3var_cross_comparison.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

## 9. Pooled vs Isolated -- Key IRF Comparison

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))

arch_colors = {'Pooled': '#2ecc71', 'Isolated': '#e67e22'}

for ax, (imp, resp, title) in zip(axes.flat, cross_irfs):
    for label, color in arch_colors.items():
        res = all_results[label]
        vals = res['irf_orth'][:, imp, resp]
        periods = range(len(vals))
        ax.plot(periods, vals, color=color, lw=2.5, label=f'{label} (N={res["n_csus"]})')
        if res['ci_lo'] is not None:
            ax.fill_between(periods,
                            res['ci_lo'][:, imp, resp],
                            res['ci_hi'][:, imp, resp],
                            alpha=0.15, color=color)
    ax.axhline(0, color='black', ls='--', lw=0.5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Days')
    ax.legend(fontsize=9)

fig.suptitle('Pooled vs Isolated Architecture: 3-Variable IRF Comparison',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/irf_3var_pooled_vs_isolated.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

## 10. IRF Summary Table

In [ ]:
irf_rows = []
for label, res in all_results.items():
    for imp, resp, name in cross_irfs:
        vals = res['irf_orth'][:, imp, resp]
        peak_idx = int(np.argmax(np.abs(vals)))
        cumul = np.sum(vals)
        
        # Count significant horizons
        n_sig = 0
        if res['ci_lo'] is not None:
            for h in range(len(vals)):
                lo = res['ci_lo'][h, imp, resp]
                hi = res['ci_hi'][h, imp, resp]
                if lo > 0 or hi < 0:
                    n_sig += 1
        
        irf_rows.append({
            'Sample': label,
            'IRF Path': name,
            'Impact (t=1)': f'{vals[1]:.5f}',
            'Peak': f'{vals[peak_idx]:.5f}',
            'Peak Period': f't+{peak_idx}',
            'Cumulative': f'{cumul:.5f}',
            'Sign': '+' if cumul > 0 else '-',
            'Sig Horizons': f'{n_sig}/21',
        })

irf_df = pd.DataFrame(irf_rows)
print("Table 2: IRF Summary -- All Cross-Variable Paths")
print("=" * 110)
# Print grouped by sample
for label in all_results:
    sub = irf_df[irf_df['Sample'] == label]
    print(f"\n--- {label} ---")
    print(sub.drop(columns='Sample').to_string(index=False))

## 11. Detailed IRF Values with Significance -- Key Paths

In [ ]:
key_paths = [
    ('Full Sample', 1, 2, 'Vol -> Liq'),
    ('Full Sample', 0, 2, 'Util -> Liq'),
    ('Full Sample', 2, 0, 'Liq -> Util'),
    ('Full Sample', 0, 1, 'Util -> Vol'),
    ('Pooled',      1, 2, 'Vol -> Liq'),
    ('Isolated',    1, 2, 'Vol -> Liq'),
]

for sample, imp, resp, name in key_paths:
    res = all_results[sample]
    vals = res['irf_orth'][:, imp, resp]
    
    print(f"\n{'='*60}")
    print(f"  {sample}: {name}")
    print(f"{'='*60}")
    print(f"{'Horizon':>8}  {'IRF':>10}  {'95% CI':>24}  {'Sig':>4}")
    print(f"{'-'*52}")
    
    for h in range(min(21, len(vals))):
        ci_str = ''
        sig = ''
        if res['ci_lo'] is not None:
            lo = res['ci_lo'][h, imp, resp]
            hi = res['ci_hi'][h, imp, resp]
            ci_str = f'[{lo:9.6f}, {hi:9.6f}]'
            if lo > 0 or hi < 0:
                sig = ' *'
        print(f"  t+{h:2d}    {vals[h]:10.6f}  {ci_str}  {sig}")

## 12. Forecast Error Variance Decomposition (FEVD)

In [ ]:
res = all_results['Full Sample']
fevd = res['results'].fevd(20)

print("Forecast Error Variance Decomposition (Full Sample)")
print("=" * 70)

var_names_fevd = ['Utilization', 'Volatility', 'Liquidation']
for resp_idx, resp_name in enumerate(var_names_fevd):
    print(f"\n--- {resp_name} forecast error explained by: ---")
    print(f"{'Horizon':>8}  {'Util':>8}  {'Vol':>8}  {'Liq':>8}")
    print(f"{'-'*36}")
    decomp = fevd.decomp  # shape: (periods, n_vars, n_vars)
    for h in [0, 1, 2, 5, 10, 20]:
        if h < decomp.shape[0]:
            vals = decomp[h, resp_idx, :]
            print(f"  t+{h:2d}    {vals[0]:7.1%}  {vals[1]:7.1%}  {vals[2]:7.1%}")

## 13. Architecture Comparison Table

In [ ]:
comp_rows = []
for label in ['Full Sample', 'Pooled', 'Isolated', 'Stablecoin', 'Volatile Asset']:
    res = all_results[label]
    
    # Cumulative IRFs for key paths
    irf_uv = np.sum(res['irf_orth'][:, 0, 1])  # util -> vol
    irf_ul = np.sum(res['irf_orth'][:, 0, 2])  # util -> liq
    irf_vl = np.sum(res['irf_orth'][:, 1, 2])  # vol -> liq
    irf_lu = np.sum(res['irf_orth'][:, 2, 0])  # liq -> util
    irf_lv = np.sum(res['irf_orth'][:, 2, 1])  # liq -> vol
    irf_vu = np.sum(res['irf_orth'][:, 1, 0])  # vol -> util
    
    comp_rows.append({
        'Sample': label,
        'N': res['n_csus'],
        'Obs': f"{res['n_obs']:,}",
        'Lags': res['lags'],
        'Cum U->V': f"{irf_uv:.5f}",
        'Cum V->L': f"{irf_vl:.5f}",
        'Cum U->L': f"{irf_ul:.5f}",
        'Cum L->U': f"{irf_lu:.5f}",
        'Cum L->V': f"{irf_lv:.5f}",
        'Cum V->U': f"{irf_vu:.5f}",
    })

comp_df = pd.DataFrame(comp_rows)
print("Table 3: Architecture Comparison -- Cumulative IRFs")
print("=" * 130)
print(comp_df.to_string(index=False))

## 14. Robustness Check: Alternative Cholesky Ordering

The baseline ordering is **Utilization -> Volatility -> Liquidation** (slow to fast).

Here we re-estimate with the alternative ordering **Utilization -> Liquidation -> Volatility**, which assumes liquidations are protocol-mechanical events that then feed into market volatility, rather than volatility triggering liquidations within the day.

**What changes:** Only the contemporaneous (t=0) restrictions on orthogonalized IRFs and the FEVD allocation of shared variance.

In [ ]:
# Alternative ordering: Utilization -> Liquidation -> Volatility
ALT_FE_COLS = ['util_fe', 'liq_fe', 'vol_fe']  # swapped liq and vol
ALT_VAR_NAMES = ['Utilization', 'Liquidation', 'Volatility']

def run_panel_var_alt_ordering(df, csus, label, max_lags=10, irf_periods=20, n_boot=500):
    """Re-estimate with alternative Cholesky ordering: util -> liq -> vol."""
    sub = df[df['csu'].isin(csus)].copy()
    sub = sub.dropna(subset=VAR_COLS)
    
    for orig, fe in zip(VAR_COLS, FE_COLS):
        sub[fe] = sub.groupby('csu')[orig].transform(lambda x: x - x.mean())
    
    # Reorder columns for Cholesky
    panel = sub[ALT_FE_COLS].dropna()
    
    model = VAR(panel)
    try:
        lo = model.select_order(maxlags=min(max_lags, len(panel) // 10))
        optimal = max(1, min(lo.bic, 5))
    except:
        optimal = 2
    
    results = model.fit(optimal)
    irf = results.irf(irf_periods)
    irf_orth = irf.orth_irfs
    
    # Bootstrap CIs
    rng = np.random.RandomState(42)
    boot_irfs = []
    resids = results.resid.values
    fitted = results.fittedvalues.values
    
    for _ in range(n_boot):
        idx = rng.choice(len(resids), size=len(resids), replace=True)
        boot_data = fitted + resids[idx]
        try:
            br = VAR(pd.DataFrame(boot_data, columns=panel.columns)).fit(optimal)
            boot_irfs.append(br.irf(irf_periods).orth_irfs)
        except:
            continue
    
    ci_lo = ci_hi = None
    if len(boot_irfs) > 50:
        ba = np.array(boot_irfs)
        ci_lo = np.percentile(ba, 2.5, axis=0)
        ci_hi = np.percentile(ba, 97.5, axis=0)
    
    return {
        'label': label,
        'results': results,
        'irf_orth': irf_orth,
        'ci_lo': ci_lo,
        'ci_hi': ci_hi,
        'lags': optimal,
    }

# Run alternative ordering for all subsamples
alt_results = {}
for label, csus in samples.items():
    alt_results[label] = run_panel_var_alt_ordering(df_clean, csus, label)
    print(f"  {label}: done (lags={alt_results[label]['lags']})")

print("\nAlternative ordering estimated for all subsamples.")

### 14a. Compare IRFs: Baseline vs Alternative Ordering

In the alternative ordering, the index mapping is: 0=Utilization, 1=Liquidation, 2=Volatility.
We compare the key cross-variable IRF paths side by side.

In [ ]:
# Map from economic IRF path to (imp_idx, resp_idx) under each ordering
# Baseline: [util=0, vol=1, liq=2]
# Alternative: [util=0, liq=1, vol=2]

path_map = {
    'Util -> Vol':  {'baseline': (0, 1), 'alt': (0, 2)},
    'Util -> Liq':  {'baseline': (0, 2), 'alt': (0, 1)},
    'Vol -> Util':  {'baseline': (1, 0), 'alt': (2, 0)},
    'Vol -> Liq':   {'baseline': (1, 2), 'alt': (2, 1)},
    'Liq -> Util':  {'baseline': (2, 0), 'alt': (1, 0)},
    'Liq -> Vol':   {'baseline': (2, 1), 'alt': (1, 2)},
}

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
path_names = list(path_map.keys())

for ax, pname in zip(axes.flat, path_names):
    bi, bj = path_map[pname]['baseline']
    ai, aj = path_map[pname]['alt']
    
    # Full Sample only
    base_vals = all_results['Full Sample']['irf_orth'][:, bi, bj]
    alt_vals = alt_results['Full Sample']['irf_orth'][:, ai, aj]
    periods = range(len(base_vals))
    
    ax.plot(periods, base_vals, color='#2c3e50', lw=2.5, label='Util->Vol->Liq (baseline)')
    ax.plot(periods, alt_vals, color='#e74c3c', lw=2.5, ls='--', label='Util->Liq->Vol (alternative)')
    
    # CIs for baseline
    res_b = all_results['Full Sample']
    if res_b['ci_lo'] is not None:
        ax.fill_between(periods, res_b['ci_lo'][:, bi, bj], res_b['ci_hi'][:, bi, bj],
                        alpha=0.12, color='#2c3e50')
    # CIs for alt
    res_a = alt_results['Full Sample']
    if res_a['ci_lo'] is not None:
        ax.fill_between(periods, res_a['ci_lo'][:, ai, aj], res_a['ci_hi'][:, ai, aj],
                        alpha=0.12, color='#e74c3c')
    
    ax.axhline(0, color='black', ls='--', lw=0.5)
    ax.set_title(pname, fontsize=13, fontweight='bold')
    ax.set_xlabel('Days')
    ax.legend(fontsize=8)

fig.suptitle('Robustness: Baseline vs Alternative Cholesky Ordering (Full Sample)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/irf_3var_ordering_robustness.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

### 14b. Cumulative IRF Comparison Table

In [ ]:
# Compare cumulative IRFs and significance across orderings for all subsamples
print("Table 4: Robustness -- Cumulative IRFs by Cholesky Ordering")
print("=" * 120)
print(f"{'Sample':<15} {'IRF Path':<12} {'Baseline Cum':>14} {'Alt Cum':>14} {'Diff':>10} {'Base Sig':>10} {'Alt Sig':>10}")
print("-" * 120)

for label in ['Full Sample', 'Pooled', 'Isolated', 'Stablecoin', 'Volatile Asset']:
    for pname in path_names:
        bi, bj = path_map[pname]['baseline']
        ai, aj = path_map[pname]['alt']
        
        base_vals = all_results[label]['irf_orth'][:, bi, bj]
        alt_vals = alt_results[label]['irf_orth'][:, ai, aj]
        
        base_cum = np.sum(base_vals)
        alt_cum = np.sum(alt_vals)
        diff = alt_cum - base_cum
        
        # Count significant horizons
        def count_sig(res, i, j):
            n = 0
            if res['ci_lo'] is not None:
                for h in range(res['irf_orth'].shape[0]):
                    if res['ci_lo'][h, i, j] > 0 or res['ci_hi'][h, i, j] < 0:
                        n += 1
            return n
        
        base_sig = count_sig(all_results[label], bi, bj)
        alt_sig = count_sig(alt_results[label], ai, aj)
        
        print(f"{label:<15} {pname:<12} {base_cum:>14.5f} {alt_cum:>14.5f} {diff:>10.5f} {base_sig:>7}/21 {alt_sig:>7}/21")
    print()

print("Positive Diff = alternative ordering gives larger cumulative response")
print("Sig Horizons = number of periods where 95% CI excludes zero")

## 15. Mechanism Comparison: Aave-Style (Instant) vs Compound V3 (Absorb)

**Aave-style** protocols allow any third-party liquidator to immediately seize discounted collateral when health factor < 1. This creates a competitive, MEV-driven liquidation market.

**Compound V3 (absorb)** uses a two-step model: the protocol itself absorbs the underwater position, then sells collateral via its own mechanism. This centralizes liquidation execution within the protocol.

We compare the dynamic responses across these two mechanism types using the same 3-variable PVAR framework.

In [ ]:
# === Liquidation Mechanism Subsamples ===

aave_style_csus = [
    'aave_v3_ethereum', 'aave_v3_base', 'aave_v3_arbitrum',
    'aave_v3_optimism', 'aave_v3_polygon', 'aave_v3_avalanche',
    'aave_v3_binance', 'aave_v3_linea', 'aave_v3_scroll', 'aave_v3_xdai',
    'moonwell_lending_base', 'sparklend_ethereum',
    'benqi_lending_avalanche', 'venus_core_pool_binance',
    'fluid_lending_arbitrum',
    'compound_v2_ethereum',
    'lodestar_lending_arbitrum', 'mendi_lending_linea',
]

compound_v3_csus = isolated_csus  # same as isolated

# Filter to CSUs present in clean data
aave_style_csus = [c for c in aave_style_csus if c in valid]
compound_v3_csus = [c for c in compound_v3_csus if c in valid]

mech_samples = {
    'Aave-Style (Instant)': aave_style_csus,
    'Compound V3 (Absorb)': compound_v3_csus,
    'Full Sample': pooled_csus + isolated_csus,
}

print("Liquidation Mechanism Subsamples:")
print("-" * 60)
for label, csus in mech_samples.items():
    sub = df_clean[df_clean['csu'].isin(csus)]
    n_liq = sub['n_liquidations'].sum()
    print(f"  {label}: {len(csus)} CSUs, {len(sub):,} obs, {n_liq:,} liq events")

In [ ]:
# Estimate Panel VAR by Mechanism
mech_results = {}
for label, csus in mech_samples.items():
    print(f"\n{'='*60}")
    print(f"  {label} ({len(csus)} CSUs)")
    print(f"{'='*60}")
    mech_results[label] = run_panel_var_3v(df_clean, csus, label)

In [ ]:
# IRF Plots -- Aave-Style vs Compound V3
mech_colors = {
    'Aave-Style (Instant)': '#e74c3c',
    'Compound V3 (Absorb)': '#3498db',
    'Full Sample': 'black',
}

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
for ax, (imp, resp, title) in zip(axes.flat, cross_irfs):
    for label, color in mech_colors.items():
        res = mech_results[label]
        vals = res['irf_orth'][:, imp, resp]
        periods = range(len(vals))
        lw = 2.5 if label == 'Full Sample' else 2.0
        ls = '-' if label != 'Full Sample' else '--'
        ax.plot(periods, vals, color=color, lw=lw, ls=ls,
                label=f'{label} (N={res["n_csus"]})')
        if res['ci_lo'] is not None and label != 'Full Sample':
            ax.fill_between(periods,
                            res['ci_lo'][:, imp, resp],
                            res['ci_hi'][:, imp, resp],
                            alpha=0.15, color=color)
    ax.axhline(0, color='black', ls=':', lw=0.8)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Days')
    ax.legend(fontsize=8)

fig.suptitle('Liquidation Mechanism Comparison: Aave-Style vs Compound V3 (3-Variable PVAR)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/irf_3var_mechanism_comparison.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()

## 16. Mechanism IRF Summary Table

In [ ]:
mech_irf_rows = []
for label, res in mech_results.items():
    for imp, resp, name in cross_irfs:
        vals = res['irf_orth'][:, imp, resp]
        peak_idx = int(np.argmax(np.abs(vals)))
        cumul = np.sum(vals)
        
        n_sig = 0
        if res['ci_lo'] is not None:
            for h in range(len(vals)):
                lo = res['ci_lo'][h, imp, resp]
                hi = res['ci_hi'][h, imp, resp]
                if lo > 0 or hi < 0:
                    n_sig += 1
        
        mech_irf_rows.append({
            'Mechanism': label,
            'IRF Path': name,
            'Impact (t=1)': f'{vals[1]:.5f}',
            'Peak': f'{vals[peak_idx]:.5f}',
            'Peak Period': f't+{peak_idx}',
            'Cumulative': f'{cumul:.5f}',
            'Sign': '+' if cumul > 0 else '-',
            'Sig Horizons': f'{n_sig}/21',
        })

mech_irf_df = pd.DataFrame(mech_irf_rows)
print("Table 5: IRF Summary -- Liquidation Mechanism Comparison")
print("=" * 110)
for label in mech_results:
    sub = mech_irf_df[mech_irf_df['Mechanism'] == label]
    res = mech_results[label]
    print(f"\n--- {label} (N={res['n_csus']}, {res['n_obs']:,} obs, {res['lags']} lags) ---")
    print(sub.drop(columns='Mechanism').to_string(index=False))

## 17. Export Results

In [ ]:
import os
results_dir = '../results'
os.makedirs(results_dir, exist_ok=True)

# Table 1: Panel Summary Statistics
summary_df.to_csv(f'{results_dir}/table1_panel_summary_stats.csv', index=False)

# Table 2: IRF Summary (baseline ordering)
irf_df.to_csv(f'{results_dir}/table2_irf_summary_baseline.csv', index=False)

# Table 3: Architecture Comparison
comp_df.to_csv(f'{results_dir}/table3_architecture_comparison.csv', index=False)

# Table 5: Mechanism IRF Summary
mech_irf_df.to_csv(f'{results_dir}/table5_mechanism_irf_summary.csv', index=False)

# IRF point estimates for all subsamples
detail_rows = []
for label, res in all_results.items():
    for imp, resp, name in cross_irfs:
        vals = res['irf_orth'][:, imp, resp]
        for h in range(len(vals)):
            row = {'Sample': label, 'IRF Path': name, 'Horizon': h, 'IRF': vals[h]}
            if res['ci_lo'] is not None:
                row['CI_lo'] = res['ci_lo'][h, imp, resp]
                row['CI_hi'] = res['ci_hi'][h, imp, resp]
                row['Significant'] = (row['CI_lo'] > 0) or (row['CI_hi'] < 0)
            detail_rows.append(row)
pd.DataFrame(detail_rows).to_csv(f'{results_dir}/irf_point_estimates.csv', index=False)

# Mechanism IRF point estimates
mech_detail = []
for label, res in mech_results.items():
    for imp, resp, name in cross_irfs:
        vals = res['irf_orth'][:, imp, resp]
        for h in range(len(vals)):
            row = {'Mechanism': label, 'IRF Path': name, 'Horizon': h, 'IRF': vals[h]}
            if res['ci_lo'] is not None:
                row['CI_lo'] = res['ci_lo'][h, imp, resp]
                row['CI_hi'] = res['ci_hi'][h, imp, resp]
                row['Significant'] = (row['CI_lo'] > 0) or (row['CI_hi'] < 0)
            mech_detail.append(row)
pd.DataFrame(mech_detail).to_csv(f'{results_dir}/mechanism_irf_point_estimates.csv', index=False)

# List saved files
saved = sorted(os.listdir(results_dir))
print("Results exported to ../results/:")
print("-" * 60)
for f in saved:
    size = os.path.getsize(f'{results_dir}/{f}')
    print(f"  {f} ({size:,} bytes)")